[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoChienHung/Latent/blob/main/notebooks/Stage_7_Sentiment_Analysis.ipynb)\n\n---\n

In [ ]:
# ══════════════════════════════════════════
# Environment Setup (Auto-Detect Colab / Local)
# ══════════════════════════════════════════
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running on Google Colab. Setting up environment...')
    from google.colab import drive
    drive.mount('/content/drive')
    
    PROJECT_ROOT = '/content/drive/MyDrive/Latent'
    os.makedirs(PROJECT_ROOT, exist_ok=True)
    os.makedirs(f'{PROJECT_ROOT}/intermediate_data', exist_ok=True)
    os.makedirs(f'{PROJECT_ROOT}/utilities', exist_ok=True)
    
    if not os.path.exists('/content/Latent'):
        !git clone https://github.com/ChaoChienHung/Latent.git /content/Latent
        %cd /content/Latent
    else:
        %cd /content/Latent
        !git pull
        
    !pip install -q -r requirements.txt
    !python -m spacy download en_core_web_sm -q
    print(f'Working directory: {os.getcwd()}')
    print(f'Data directory: {PROJECT_ROOT}')
else:
    PROJECT_ROOT = '.'
    print('Running locally. Ensure dependencies are installed via: pip install -r requirements.txt')
\n

# Stage 7: Sentiment & Emotion Analysis

This stage performs emotion classification on the preprocessed Reddit posts using
a pretrained DistilRoBERTa model (`j-hartmann/emotion-english-distilroberta-base`).

### 7-Class Emotion Categories
- anger, disgust, fear, joy, neutral, sadness, surprise

### Pipeline Position
- **Input**: `intermediate_data/PostVault.csv` (from Stage 5)
- **Output**: `emotion_output/PostVault_labels.csv` (with emotion predictions)


In [ ]:
# Install dependencies (uncomment if needed)
# !pip install torch transformers pandas tqdm


## 7.1 Setup & Configuration


In [ ]:
import os
import sys
import logging
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

# Configuration
MODEL_NAME = "j-hartmann/emotion-english-distilroberta-base"
INPUT_CSV = "intermediate_data/PostVault.csv"
OUTPUT_DIR = "emotion_output"
BATCH_SIZE = 16  # Lower if you hit GPU memory limits
MAX_LENGTH = 256

# Auto-detect device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


## 7.2 Load Model & Tokenizer


In [ ]:
print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

# Get label mapping
id2label = model.config.id2label
labels = [id2label[i] for i in range(model.config.num_labels)]
print(f"Emotion labels: {labels}")


## 7.3 Load Data


In [ ]:
df = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"Loaded {len(df)} rows from {INPUT_CSV}")
print(f"Columns: {list(df.columns)}")

# Choose text column
TEXT_PRIORITY = ("cleaned_text_normalized", "cleaned_text", "raw_text", "body")
text_col = None
for col in TEXT_PRIORITY:
    if col in df.columns:
        text_col = col
        break

if text_col is None and {"title", "selftext"}.issubset(df.columns):
    df["__text__"] = (df["title"].fillna("") + " " + df["selftext"].fillna("")).str.strip()
    text_col = "__text__"

print(f"Using text column: {text_col}")
df["_text"] = df[text_col].fillna("").astype(str).str.strip()
valid_mask = df["_text"].ne("")
print(f"Valid texts: {valid_mask.sum()} / {len(df)}")


## 7.4 Run Emotion Classification


In [ ]:
from tqdm.auto import tqdm

texts = df.loc[valid_mask, "_text"].tolist()
all_predictions = []
all_probs = []

with torch.inference_mode():
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Classifying"):
        batch = texts[i : i + BATCH_SIZE]
        encoded = tokenizer(
            batch, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        logits = model(**encoded).logits
        probs = torch.softmax(logits, dim=-1).cpu()
        
        for row in probs:
            all_predictions.append(labels[int(row.argmax())])
            all_probs.append(row.tolist())

print(f"Classified {len(all_predictions)} texts")


## 7.5 Save Results


In [ ]:
# Add predictions to dataframe
result = df.copy()
result["predicted_emotion"] = pd.NA
for label in labels:
    result[f"prob_{label}"] = pd.NA

valid_idx = result.index[valid_mask]
result.loc[valid_idx, "predicted_emotion"] = all_predictions
for i, label in enumerate(labels):
    result.loc[valid_idx, f"prob_{label}"] = [p[i] for p in all_probs]

# Save
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, "PostVault_labels.csv")
result.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

# Quick summary
print("\nEmotion distribution:")
print(result["predicted_emotion"].value_counts())


## 7.6 Quick Analysis


In [ ]:
# Average confidence per emotion
prob_cols = [f"prob_{l}" for l in labels]
result["max_prob"] = result[prob_cols].max(axis=1)

for emotion in labels:
    sub = result[result["predicted_emotion"] == emotion]
    avg_conf = sub["max_prob"].mean()
    print(f"{emotion:>10}: {len(sub):5d} posts, avg confidence: {avg_conf:.3f}")
